# CS514 BGG Data Collection Walkthrough

This notebook documents the data-collection process behind the CS514 BoardGameGeek network project. It is designed as a descriptive walkthrough: what endpoints were used, what raw responses looked like, how candidate users became reliable users, why the diversity expansion was added, and which processed artifacts resulted from each phase.

This is not the main network-analysis notebook. Its purpose is to explain and verify the data pipeline.

## High-Level Pipeline

1. Build ranked game universe from BGG ranked game data.
2. Collect detailed metadata for the top 5,000 ranked games.
3. Select the top 2,500 ranked games as the CS514 network universe.
4. Parse BGG mechanics and categories to create taxonomy metadata.
5. Discover candidate users from rating comments and play logs of seed games.
6. Fetch full BGG collections for candidate users.
7. Filter fetched users into reliable users using collection-size and selected-game-overlap thresholds.
8. Diagnose underrepresented mechanics/categories in the baseline reliable-user dataset.
9. Run a targeted diversity expansion from underrepresented seed games.
10. Merge baseline and strict expansion users for network analysis.

In [ ]:
from pathlib import Path
import html
import json
import textwrap
import xml.etree.ElementTree as ET

import pandas as pd
from IPython.display import HTML, display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 140)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW = PROJECT_ROOT / "data" / "raw"
PROCESSED = PROJECT_ROOT / "data" / "processed"
BASELINE = PROCESSED / "reliable_users" / "reliable_users_batch1"
BASELINE_RAW = RAW / "reliable_users" / "reliable_users_batch1"
EXPANSION = PROCESSED / "reliable_users" / "diversity_expansion_batch1"
EXPANSION_RAW = RAW / "reliable_users" / "diversity_expansion_batch1"
DETAILS_CSV = PROCESSED / "top_ranked_games_details" / "top_ranked_games_details_top5000_ranked_only" / "top_ranked_games_details.csv"

def read_json(path):
    with open(path, "r", encoding="utf-8") as fh:
        return json.load(fh)

def first_existing(patterns):
    for pattern in patterns:
        matches = sorted(PROJECT_ROOT.glob(pattern))
        if matches:
            return matches[0]
    return None

def show_snippet(path, n=1600):
    """Display raw XML safely in notebooks and Windows terminals."""
    if path is None:
        display(HTML("<em>No matching raw file found.</em>"))
        return
    text = path.read_text(encoding="utf-8", errors="replace")[:n]
    escaped = html.escape(text)
    display(HTML(
        f"<div><strong>{html.escape(str(path))}</strong></div>"
        f"<pre style='white-space: pre-wrap; max-height: 380px; overflow: auto; border: 1px solid #ddd; padding: 8px;'>{escaped}</pre>"
    ))

print(PROJECT_ROOT)

## BGG API / XML Endpoints Used

The user-discovery and collection stages used BGG XML/API endpoints. Web scraping/parsing was used for taxonomy support, not for extracting user interactions.

The important endpoint revelation was that `collection?stats=1` includes rating information inside the full collection response. That made separate rated-only collection requests redundant for future pipelines.

In [ ]:
endpoint_rows = [
    {
        "endpoint": "thing?id=GAME_ID&ratingcomments=1&page=PAGE&pagesize=100",
        "used_for": "Candidate user discovery",
        "response_contains": "Usernames from rating-comment rows, user rating, whether comment text exists",
        "pipeline_role": "Find users who interacted with a seed game through ratings/comments",
    },
    {
        "endpoint": "plays?id=GAME_ID&type=thing&page=PAGE",
        "used_for": "Candidate user discovery",
        "response_contains": "Play records and player usernames",
        "pipeline_role": "Find users who logged plays of a seed game",
    },
    {
        "endpoint": "collection?username=USER&stats=1",
        "used_for": "User enrichment",
        "response_contains": "Collection items, status flags, numplays, user rating",
        "pipeline_role": "Turn a candidate username into full user-game edge data",
    },
    {
        "endpoint": "thing?id=GAME_IDS&stats=1",
        "used_for": "Game detail collection",
        "response_contains": "Game metadata and statistics",
        "pipeline_role": "Build detailed game-level dataset",
    },
]
pd.DataFrame(endpoint_rows)

## Example 1: Rating-Comment Response

Candidate users were discovered from rating-comment pages for seed games. Before enrichment, the data only told us that a username appeared on a game's rating-comment page. It did not yet tell us the user's full collection size or reliability.

In [ ]:
rating_xml = first_existing([
    "data/raw/reliable_users/diversity_expansion_batch1/discovery/ratingcomments*.xml",
    "data/raw/rank_range_users/*/discovery/ratingcomments*.xml",
    "data/raw/user_candidates/*/ratingcomments*.xml",
])
show_snippet(rating_xml, n=1800)

In [ ]:
def parse_rating_comment_example(path, max_rows=8):
    if path is None:
        return pd.DataFrame()
    root = ET.fromstring(path.read_text(encoding="utf-8", errors="replace"))
    rows = []
    for comment in root.findall(".//comments/comment")[:max_rows]:
        rows.append({
            "username": comment.attrib.get("username"),
            "rating": comment.attrib.get("rating"),
            "comment_has_text": bool((comment.attrib.get("value") or "").strip()),
            "comment_preview": (comment.attrib.get("value") or "")[:100],
        })
    return pd.DataFrame(rows)

parse_rating_comment_example(rating_xml)

## Example 2: Plays Response

Play logs were the second discovery source. A user appearing in play logs is a stronger activity signal than a one-off rating comment because it indicates logged play behavior around the seed game.

In [ ]:
plays_xml = first_existing([
    "data/raw/reliable_users/diversity_expansion_batch1/discovery/plays*.xml",
    "data/raw/rank_range_users/*/discovery/plays*.xml",
    "data/raw/user_candidates/*/plays*.xml",
])
show_snippet(plays_xml, n=1800)

In [ ]:
def parse_plays_example(path, max_rows=10):
    if path is None:
        return pd.DataFrame()
    root = ET.fromstring(path.read_text(encoding="utf-8", errors="replace"))
    rows = []
    for play in root.findall("play"):
        for player in play.findall("players/player"):
            username = player.attrib.get("username")
            if username:
                rows.append({
                    "play_id": play.attrib.get("id"),
                    "play_date": play.attrib.get("date"),
                    "quantity": play.attrib.get("quantity"),
                    "username": username,
                })
            if len(rows) >= max_rows:
                return pd.DataFrame(rows)
    return pd.DataFrame(rows)

parse_plays_example(plays_xml)

## Example 3: Full User Collection Response

After candidate discovery, we fetched each candidate user's collection. This is the step that turned usernames into actual user-game edges.

Collection rows include status flags such as `own`, `wishlist`, `wanttoplay`, `prevowned`, and `fortrade`, plus `numplays` and, when `stats=1` is used, `user_rating`.

In [ ]:
collection_xml = first_existing([
    "data/raw/reliable_users/diversity_expansion_batch1/collections/*collection_all_stats.xml",
    "data/raw/reliable_users/reliable_users_batch1/collections/*collection_all.xml",
])
show_snippet(collection_xml, n=2200)

In [ ]:
def parse_collection_example(path, max_rows=10):
    if path is None:
        return pd.DataFrame()
    root = ET.fromstring(path.read_text(encoding="utf-8", errors="replace"))
    rows = []
    for item in root.findall("item")[:max_rows]:
        status = item.find("status")
        rating = item.find("stats/rating")
        rows.append({
            "bgg_id": item.attrib.get("objectid"),
            "name": item.findtext("name"),
            "own": status.attrib.get("own") if status is not None else None,
            "wishlist": status.attrib.get("wishlist") if status is not None else None,
            "wanttoplay": status.attrib.get("wanttoplay") if status is not None else None,
            "prevowned": status.attrib.get("prevowned") if status is not None else None,
            "fortrade": status.attrib.get("fortrade") if status is not None else None,
            "numplays": item.findtext("numplays"),
            "user_rating": rating.attrib.get("value") if rating is not None else None,
        })
    return pd.DataFrame(rows)

parse_collection_example(collection_xml)

## Game Metadata And Selected Game Universe

The detailed game file contains the game-level metadata. We collected detailed data for 5,000 ranked games, then used a fixed top-2,500 selected game universe for CS514 network analysis.

In [ ]:
games = pd.read_csv(DETAILS_CSV)
selected_games = pd.read_csv(BASELINE / "selected_games.csv")

def split_pipe(series):
    values = []
    for item in series.dropna().astype(str):
        values.extend([part.strip() for part in item.split("|") if part.strip()])
    return values

summary = pd.DataFrame([
    {"metric": "detailed ranked games", "value": len(games)},
    {"metric": "selected CS514 games", "value": len(selected_games)},
    {"metric": "unique mechanics in detailed games", "value": len(set(split_pipe(games.get("mechanics", pd.Series(dtype=str)))))} ,
    {"metric": "unique categories in detailed games", "value": len(set(split_pipe(games.get("categories", pd.Series(dtype=str)))))} ,
])
summary

In [ ]:
selected_games.head(10)

## Baseline Reliable-User Pipeline

The baseline pipeline discovered candidate users from high-ranked seed games and fetched full collections for those candidates. Users became reliable only after enrichment and filtering.

Important correction: we did not know collection size before collection API calls. Before enrichment, users were only candidates.

In [ ]:
baseline_meta = read_json(BASELINE / "metadata.json")
baseline_thresholds = read_json(BASELINE / "selection_thresholds.json")

pd.DataFrame([
    {"field": "target reliable users", "value": baseline_meta["target_established_users"]},
    {"field": "candidate users", "value": baseline_meta["candidate_user_count"]},
    {"field": "fully fetched users", "value": baseline_meta["fully_fetched_user_count"]},
    {"field": "reliable users", "value": baseline_meta["established_user_count"]},
    {"field": "discovery rating rows", "value": baseline_meta["discovery_rating_row_count"]},
    {"field": "discovery play rows", "value": baseline_meta["discovery_play_row_count"]},
    {"field": "discovery proceeded through rank", "value": baseline_meta["next_start_rank"] - 1},
])

In [ ]:
baseline_thresholds

### What Was Known Before User Collection Fetches?

The candidate-user table only had discovery information: username, how many times the user was discovered, which source games/ranks they came from, and whether the source was rating comments or play logs. It did not yet contain full collection information.

In [ ]:
candidate_users = pd.read_csv(BASELINE / "all_candidate_users.csv")
candidate_users.head(10)

### Reliable User Rule

A baseline user was kept as reliable if:

- collection fetch succeeded
- `collection_item_count >= 50`
- `selected_game_overlap_count >= 10`

`collection_item_count` means collection records across statuses, not only currently owned games. `selected_game_overlap_count` means how many of the user's collection records overlapped with the selected top-2,500 game universe.

In [ ]:
reliable_users = pd.read_csv(BASELINE / "reliable_users.csv")
reliable_users[[
    "username",
    "collection_item_count",
    "owned_count",
    "rated_count",
    "wishlist_count",
    "prevowned_count",
    "selected_game_overlap_count",
    "selected_owned_count",
    "selected_rated_count",
]].head(10)

In [ ]:
reliable_users[[
    "collection_item_count",
    "owned_count",
    "rated_count",
    "selected_game_overlap_count",
    "selected_owned_count",
    "selected_rated_count",
]].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99]).T

## Ratings Backfill And The `stats=1` Revelation

During implementation it was discovered that ratings did not require a separate rated-only collection endpoint. Instead, the full collection endpoint with `stats=1` includes rating values for collection items.

Practical conclusion:

- `collection?username=USER&stats=1` is sufficient for ownership/status plus ratings.
- Separate `collection?username=USER&rated=1&stats=1` fetches are redundant for future pipelines.

The baseline was backfilled so the existing reliable-user edges gained usable rating data.

In [ ]:
backfill_meta_path = BASELINE / "user_ratings_backfill_metadata.json"
if backfill_meta_path.exists():
    display(read_json(backfill_meta_path))

edges_sample = pd.read_csv(BASELINE / "reliable_user_collection_edges.csv", nrows=20000)
pd.DataFrame([
    {"metric": "sample rows", "value": len(edges_sample)},
    {"metric": "rows with user_rating in sample", "value": edges_sample["user_rating"].notna().sum()},
    {"metric": "rows with own=1 in sample", "value": (edges_sample["own"] == 1).sum()},
    {"metric": "rows with wishlist=1 in sample", "value": (edges_sample["wishlist"] == 1).sum()},
    {"metric": "rows with prevowned=1 in sample", "value": (edges_sample["prevowned"] == 1).sum()},
])

## Baseline Bias Diagnosis

The baseline was large and dense, but it was discovered mostly from very high-ranked games. This produced a mainstream/high-visibility engaged-user bias. We diagnosed this by checking mechanics/category coverage and identifying tags with weak committed-user coverage.

In [ ]:
mechanics_cov = pd.read_csv(BASELINE / "taxonomy_coverage_mechanics.csv")
categories_cov = pd.read_csv(BASELINE / "taxonomy_coverage_categories.csv")

display(mechanics_cov.head(10))
display(categories_cov.head(10))

## Diversity Expansion

The second sweep was not run to reach 5,000 users. The baseline had already reached that target. The diversity expansion was added after bias diagnosis to improve coverage of underrepresented BGG communities.

The expansion remained isolated from the baseline and used new seed games selected by:

1. taxonomy-deficit tags,
2. rank-stratified seeds,
3. wishlist-ratio contrast seeds.

Candidate users were again discovered from rating comments and play logs, but the seed games were chosen to target weak communities.

In [ ]:
expansion_meta = read_json(EXPANSION / "metadata.json")
expansion_thresholds = read_json(EXPANSION / "selection_thresholds.json")
expansion_eval = read_json(EXPANSION / "diversity_evaluation.json")

pd.DataFrame([
    {"field": "baseline reliable users", "value": expansion_meta["baseline_reliable_user_count"]},
    {"field": "seed games", "value": expansion_meta["seed_count"]},
    {"field": "strict expansion users", "value": expansion_eval["expansion_metrics"]["n_users"]},
    {"field": "merged users", "value": expansion_eval["merged_metrics"]["n_users"]},
    {"field": "target tags improved >=20pct", "value": expansion_eval["target_tag_summary"]["tags_with_20pct_improvement"]},
    {"field": "target tag count", "value": expansion_eval["target_tag_summary"]["target_tag_count"]},
    {"field": "stop reason", "value": (EXPANSION / "SCRIPT_STOPPED_REASON.txt").read_text(encoding="utf-8").strip()},
])

In [ ]:
seed_games = pd.read_csv(EXPANSION / "seed_games.csv")
seed_games[[
    "seed_index",
    "bgg_id",
    "name",
    "overall_rank",
    "seed_type",
    "reason_codes",
    "target_kinds",
    "target_tags",
]].head(15)

### Expansion Filters

The expansion produced two user pools:

- strict reliable pool: same style as baseline, `collection_item_count >= 50` and `selected_game_overlap >= 10`
- community pool: looser, `collection_item_count >= 20` and `selected_game_overlap >= 3`

The strict reliable pool is the main pool used in the merged CS514 analysis.

In [ ]:
{
    "tight_reliable_pool": expansion_thresholds["tight_reliable_pool"],
    "community_pool": expansion_thresholds["community_pool"],
    "collection": expansion_thresholds["collection"],
}

In [ ]:
expansion_reliable = pd.read_csv(EXPANSION / "reliable_users.csv")
expansion_community = pd.read_csv(EXPANSION / "community_users.csv")

pd.DataFrame([
    {"pool": "strict reliable", "users": len(expansion_reliable)},
    {"pool": "community", "users": len(expansion_community)},
])

## What Improved After Expansion?

The expansion succeeded because target underrepresented tags gained committed-user coverage. This is why I've stopped collection and moved to network analysis.

In [ ]:
deltas = pd.DataFrame(expansion_eval["target_tag_deltas"])
deltas[[
    "tag_kind",
    "tag",
    "core_committed_users",
    "expansion_committed_users",
    "merged_committed_users",
    "relative_gain",
]].sort_values("relative_gain", ascending=False).head(20)

## Final Data-Collection Summary

The clean version of the data-collection story is:

1. We built a ranked BGG game universe.
2. We collected detailed metadata for the top 5,000 ranked games.
3. We fixed the CS514 graph node universe at the top 2,500 games.
4. We collected mechanics/categories as metadata for later interpretation.
5. We discovered candidate users from BGG API rating-comment and play-log endpoints.
6. Candidate users initially had only discovery information, not full collection information.
7. We enriched candidates by fetching full BGG collections.
8. Users became reliable only if their collection fetch succeeded and they passed collection-size/overlap thresholds.
9. We discovered that `collection?stats=1` includes ratings, so separate rated-only collection fetches are redundant.
10. We diagnosed baseline mainstream bias through weak taxonomy coverage.
11. We ran a targeted diversity expansion using underrepresented seed games.
12. The expansion improved 28 / 30 target tags and added 478 strict reliable users.
13. The resulting dataset is dense and behaviorally rich, but should be framed as an engaged-user dataset, not a neutral sample of all BGG users.